In [ ]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

In [ ]:
%cd capstone_project_GroupA
!git checkout colab
%cd src

In [ ]:
from datetime import datetime

from ModelFiles.LSTM.LSTMUtils import *
from ModelFiles.ModelConfigs import LSTMConfig, SEEDS
from ModelFiles.ModelEnums import LSTMModelType
from ModelFiles.ModelPlots import *

use_log_target = True
EVAL_STEP_SIZE = 48
FORECAST_LAST_STEP_ONLY = [False, True]
NUM_EPOCHS = 100
PATIENCE = 10
DEBUG = False
HORIZON_LOOKBACK = [(48, 168), (336, 336), (720, 720)]
# hidden_size, num_layers, dropout, learning_rate, num_attention_heads, batch_size
HYPERPARAMETER_COMBINATIONS = [(512, 8, 0.2, 0.00001, 12, 64), (512, 12, 0.2, 0.00001, 16, 64)]
SEEDS = [31415] # run once first to see preliminary results.
for horizon, lookback in HORIZON_LOOKBACK:
    for hidden_size, num_layers, dropout, learning_rate, num_attention_heads, batch_size in HYPERPARAMETER_COMBINATIONS:
        for forecast_last_step_only in FORECAST_LAST_STEP_ONLY:
            for seed in SEEDS:
                if seed == SEEDS[-1]:
                    save_prediction_results = True
                else:
                    save_prediction_results = False
                biLSTM_config = LSTMConfig(
                    task_id= f"biLSTM_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                    model_type= LSTMModelType.BILSTM,
                    hidden_size= hidden_size,
                    num_layers= num_layers,
                    dropout= dropout,
                    learning_rate= learning_rate,
                    batch_size= batch_size,
                    training_epochs= NUM_EPOCHS,
                    patience= PATIENCE,
                    use_mlp_head= True,
                    mlp_hidden_size= 64,
                    target_col= "TOTALDEMAND" if not use_log_target else "LOG_TOTALDEMAND",
                    used_log_target= use_log_target,
                    target_lags= [48, 336],
                    target_mas= [],
                    feature_cols= ["TEMPERATURE"],
                    feature_lag_cols= [],
                    lookback_window= lookback,
                    forecast_horizon= horizon,
                    forecast_last_step_only=forecast_last_step_only,
                    num_attention_heads= num_attention_heads,
                    weight_decay= 1e-4,
                    scale=True,
                    seed=seed,
                    save_training_log= True,
                    save_test_results= save_prediction_results,
                    eval_step_size=EVAL_STEP_SIZE,
                    debug= DEBUG,
                )
                runs_df_artifact = run_experiment(config=biLSTM_config, experiment_name=biLSTM_config.task_id)
                print("=" * 200)
                print("\n")

In [ ]:
from datetime import datetime
from ModelFiles.GroupAModels import TransformersModel
from ModelFiles.ModelConfigs import TransformersConfig, HORIZONS, SEEDS
from ModelFiles.ModelEnums import TransformerModelType
from ModelFiles.ModelPlots import *

USE_LOG_TARGET = True
CONTEXT_LENGTHS = [336, 720]
EVAL_STEP_SIZE = 48
NUM_EPOCHS = 100
PATIENCE = 10
DEBUG = False

for horizon in HORIZONS:
    for context_length in CONTEXT_LENGTHS:
        if context_length >= horizon:
            for seed in SEEDS:
                if seed == SEEDS[-1]:
                    save_prediction_results = True
                else:
                    save_prediction_results = False

                timexer_config = TransformersConfig(
                    task_id=f"timexer_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                    model=TransformerModelType.TIMEXER,
                    forecast_horizon=horizon,
                    lookback_window=context_length,
                    used_log_target=USE_LOG_TARGET,
                    target_col="LOG_TOTALDEMAND" if USE_LOG_TARGET else "TOTALDEMAND",
                    feature_cols=['TEMPERATURE', 'TEMP_SQUARED', 'IS_WEEKEND', 'demand_1_year_ago'],
                    scale=True,
                    date_col='DATETIME',
                    variate='M',
                    patch_len=16,
                    stride=16,  # TimeXer uses patch_len as stride internally
                    d_model=512,
                    num_attention_heads=8,
                    num_encoder_layers=3,
                    dim_ff=2048,
                    dropout=0.1,
                    dropout_head_fc=0.1,
                    use_gpu=True,
                    time_encoding='timeF',
                    training_epochs=NUM_EPOCHS,
                    batch_size=32,
                    learning_rate=0.0001,
                    output_attention=False,
                    lradj='type1',
                    patience=PATIENCE,
                    seed=seed,
                    eval_step_size=EVAL_STEP_SIZE,
                    save_test_results=False,#save_prediction_results,
                    debug=DEBUG,
                    save_training_log=False,#True,
                    use_norm=True,
                    activation='gelu',
                )
                timexer_model = TransformersModel(timexer_config)
                timexer_model.train_model()
                timexer_model.evaluate_model(test_mode=1)
                print("=" * 200)
                print("\n")
